# Training-image preprocessing and patient audit

Run all cells from the project directory using the **.venv-preprocess** Python environment.

- Audit images per patient before and after deduplication. Keep distinct lesions/views from the same patient.
- Treat the supplied duplicate-pair CSV as a graph, keeping one available image per connected component (including transitive duplicates). Original DICOMs are **not deleted**.
- Decode baseline-JPEG DICOM frames with Pillow's reduced-resolution JPEG decoder, convert to RGB, then resize to **256 × 256** with bilinear interpolation (stretch to a square). Other transfer syntaxes use pydicom. Reduced-resolution decoding is much faster but can differ slightly from a full-resolution decode followed by resizing.
- Store lossless `uint8` arrays of the resized images. Normalize to `[0, 1]` only when loading a batch, avoiding a ~24 GB float32 cache.
- Save cleaned metadata, patient counts, removed-image mappings, and a restartable, fingerprinted image cache under **artifacts/preprocessed_train**. A matching rerun reuses the cache; interrupted runs resume. Fingerprints cover preprocessing settings, decoder versions, input CSV contents, and source file sizes/modification times.
- Use `patient_group` for downstream splits, not random image-level splits. No augmentations or fitted normalization are applied here; fit any learned transforms on the training split only.

The existing model notebook and its older artifacts are not modified; its original loader does not automatically use this new cache.

#### Verified run

- **33,126** input images; **425** CSV-listed duplicate images excluded; **32,701** retained.
- **2,056** patients, all with multiple images both before and after deduplication. No missing source images or patient IDs.
- Retained labels: **32,120 benign**, **581 malignant**.
- Completed cache: **(32,701, 256, 256, 3)**, `uint8`, approximately **5.99 GiB**.
- Passed duplicate-graph tests, metadata/cache alignment, duplicate exclusion, sampled source-to-cache pixel equality, batch normalization, and a second cache request with **zero images left to decode**.

In [1]:
from pathlib import Path
import hashlib
import json
import os
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import pydicom
from PIL import Image, __version__ as PILLOW_VERSION
from IPython.display import display
from tqdm.auto import tqdm

ROOT = Path.cwd()
if not (ROOT / 'ISIC_2020_Training_GroundTruth.csv').is_file():
    raise FileNotFoundError('Run this notebook from the CV project directory.')
IMAGE_SIZE = (256, 256)  # (width, height); RGB uint8 on disk, divide by 255 per batch.
WORKERS = min(4, os.cpu_count() or 1)
CACHE = ROOT / 'artifacts' / 'preprocessed_train'
CACHE.mkdir(parents=True, exist_ok=True)
print('Preprocessing cache:', CACHE)
print('pydicom:', pydicom.__version__)


Preprocessing cache: /home/bhandead/Projects/CV/artifacts/preprocessed_train
pydicom: 3.0.2


/home/bhandead/Projects/CV/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Build transitive duplicate groups; do not confuse repeated patients with duplicates.
def connected_components(nodes, edges):
    parent = {node: node for node in nodes}

    def find(node):
        parent.setdefault(node, node)
        while parent[node] != node:
            parent[node] = parent[parent[node]]
            node = parent[node]
        return node

    for left, right in edges:
        left_root, right_root = find(left), find(right)
        if left_root != right_root:
            parent[max(left_root, right_root)] = min(left_root, right_root)
    return {node: find(node) for node in parent}


LABEL_PATH = ROOT / 'ISIC_2020_Training_GroundTruth.csv'
DUPLICATE_PATH = ROOT / 'ISIC_2020_Training_Duplicates.csv'
labels = pd.read_csv(LABEL_PATH)
pairs = pd.read_csv(DUPLICATE_PATH)
assert labels['image_name'].notna().all() and labels['image_name'].is_unique
assert labels['target'].isin([0, 1]).all()
assert pairs[['image_name_1', 'image_name_2']].notna().all().all()
labels = labels.sort_values('image_name').reset_index(drop=True)
labels['patient_group'] = labels['patient_id'].fillna('missing:' + labels['image_name'])
labels['source_path'] = 'train/' + labels['image_name'] + '.dcm'
source_paths = [ROOT / path for path in labels['source_path']]
source_stats = [path.stat() if path.is_file() else None for path in source_paths]
labels['source_size'] = [stat.st_size if stat else -1 for stat in source_stats]
labels['source_mtime_ns'] = [stat.st_mtime_ns if stat else -1 for stat in source_stats]
labels['available'] = labels['source_size'].ge(0)
image_groups = connected_components(labels['image_name'], pairs.itertuples(index=False, name=None))
labels['duplicate_group'] = labels['image_name'].map(image_groups)

# Conflicting targets in a supplied duplicate group require manual review.
conflicts = labels.groupby('duplicate_group')['target'].nunique()
if conflicts.gt(1).any():
    raise ValueError(f'Conflicting targets in duplicate groups: {conflicts[conflicts.gt(1)].index.tolist()}')

# If a CSV duplicate links different patient IDs, keep their remaining images in one split group.
patient_edges = []
for _, members in labels.groupby('duplicate_group', sort=False):
    patients = members['patient_group'].unique()
    patient_edges.extend((patients[0], patient) for patient in patients[1:])
patient_groups = connected_components(labels['patient_group'], patient_edges)
labels['patient_group'] = labels['patient_group'].map(patient_groups)

available = labels.loc[labels['available']].copy()
keepers = available.drop_duplicates('duplicate_group', keep='first').set_index('duplicate_group')['image_name']
labels['kept_image_name'] = labels['duplicate_group'].map(keepers)
labels['removed_duplicate'] = labels['kept_image_name'].notna() & labels['image_name'].ne(labels['kept_image_name'])
removed = labels.loc[labels['removed_duplicate']].copy()
missing = labels.loc[~labels['available']].copy()
clean = labels.loc[labels['available'] & ~labels['removed_duplicate']].copy().reset_index(drop=True)
clean.insert(0, 'cache_index', np.arange(len(clean), dtype=np.int64))
if clean.empty:
    raise ValueError('No available training images remain.')
assert clean['duplicate_group'].is_unique
assert clean['image_name'].is_unique

before_counts = labels.groupby('patient_id').size().rename('images_before')
patient_summary = pd.concat([
    before_counts,
    clean.groupby('patient_id').size().rename('images_after'),
    clean.groupby('patient_id')['target'].sum().rename('malignant_images_after'),
], axis=1).fillna(0).astype(int).reset_index()
patient_summary['multiple_images_before'] = patient_summary['images_before'].gt(1)
patient_summary['multiple_images_after'] = patient_summary['images_after'].gt(1)

# Persist audit outputs; only clean rows, in cache_index order, enter the image cache.
clean.to_csv(CACHE / 'clean_metadata.csv', index=False)
removed.to_csv(CACHE / 'removed_duplicates.csv', index=False)
missing.to_csv(CACHE / 'missing_images.csv', index=False)
patient_summary.to_csv(CACHE / 'patient_summary.csv', index=False)
patient_summary.loc[patient_summary['multiple_images_after']].to_csv(CACHE / 'patients_multiple_images.csv', index=False)
known_ids = set(labels['image_name'])
unknown_pairs = pairs.loc[~pairs['image_name_1'].isin(known_ids) | ~pairs['image_name_2'].isin(known_ids)]
unknown_pairs.to_csv(CACHE / 'duplicate_pairs_unknown_images.csv', index=False)
summary = {
    'original_images': len(labels), 'duplicate_pairs': len(pairs),
    'removed_duplicate_images': len(removed), 'missing_source_images': len(missing),
    'retained_images': len(clean), 'known_patients_before': int(labels['patient_id'].nunique()),
    'known_patients_after': int(clean['patient_id'].nunique()),
    'patients_with_multiple_images_before': int(patient_summary['multiple_images_before'].sum()),
    'patients_with_multiple_images_after': int(patient_summary['multiple_images_after'].sum()),
    'missing_patient_ids': int(labels['patient_id'].isna().sum()),
    'duplicate_patient_links': len(patient_edges), 'duplicate_pairs_with_unknown_images': len(unknown_pairs),
}
print(json.dumps(summary, indent=2))
display(patient_summary.sort_values('images_after', ascending=False).head(15))
display(clean.groupby('benign_malignant').size().rename('retained_images'))

{
  "original_images": 33126,
  "duplicate_pairs": 425,
  "removed_duplicate_images": 425,
  "missing_source_images": 0,
  "retained_images": 32701,
  "known_patients_before": 2056,
  "known_patients_after": 2056,
  "patients_with_multiple_images_before": 2056,
  "patients_with_multiple_images_after": 2056,
  "missing_patient_ids": 0,
  "duplicate_patient_links": 0,
  "duplicate_pairs_with_unknown_images": 0
}


,patient_id,images_before,images_after,malignant_images_after,multiple_images_before,multiple_images_after
921,IP_4479736,115,115,0,True,True
900,IP_4382720,115,115,0,True,True
1453,IP_7279968,115,115,0,True,True
998,IP_4938382,115,115,1,True,True
114,IP_0656529,114,114,0,True,True
907,IP_4419570,102,102,0,True,True
1242,IP_6245507,102,102,0,True,True
607,IP_3057277,102,102,0,True,True
1860,IP_9147454,102,102,0,True,True
209,IP_1139701,102,102,1,True,True


benign_malignant
benign       32120
malignant      581
Name: retained_images, dtype: int64

## Linear melanoma model
 Linear model predicts the lable using the 3 channel RGB pixel values and the Logistic regression. .joblib file contains the predicted labels of the logistic regression model.